# CNN — Intel Image Classification

## Setup (Google Colab)

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Tubes2ML_KicauMania'
else:
    BASE = os.path.dirname(os.path.abspath('__file__'))

sys.path.insert(0, os.path.join(BASE, 'src'))
os.chdir(BASE)
print('Working dir:', os.getcwd())


In [ ]:
import glob, json, itertools
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import f1_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, AveragePooling2D,
                                     Flatten, Dense)
from tensorflow.keras.callbacks import EarlyStopping

os.makedirs('saved_models', exist_ok=True)

IMG_SIZE   = (64, 64)
EPOCHS     = 20
BATCH_SIZE = 32
CLASSES    = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
CLASS_MAP  = {c: i for i, c in enumerate(CLASSES)}


## Load Dataset

In [ ]:
TRAIN_DIR = os.path.join(BASE, 'CNN_dataset', 'seg_train')
TEST_DIR  = os.path.join(BASE, 'CNN_dataset', 'seg_test')

def collect_paths(root):
    paths, labels = [], []
    for cls in CLASSES:
        d = os.path.join(root, cls)
        if not os.path.isdir(d): continue
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            for fp in glob.glob(os.path.join(d, ext)):
                paths.append(fp); labels.append(CLASS_MAP[cls])
    return paths, labels

def load_dataset(paths, labels, desc='Loading'):
    N = len(paths)
    X = np.zeros((N, IMG_SIZE[0], IMG_SIZE[1], 3), dtype=np.float32)
    for i, fp in enumerate(paths):
        img = Image.open(fp).convert('RGB').resize(IMG_SIZE)
        X[i] = np.array(img) / 255.0
        if (i + 1) % 500 == 0 or (i + 1) == N:
            pct = (i + 1) / N * 100
            bar = int(pct // 5)
            print(f'\r{desc}: [' + '#' * bar + '-' * (20 - bar) + f'] {pct:5.1f}% ({i+1}/{N})', end='', flush=True)
    print()
    return X, np.array(labels, dtype=np.int32)

train_paths, train_labels = collect_paths(TRAIN_DIR)
test_paths,  test_labels  = collect_paths(TEST_DIR)
print(f'Train: {len(train_paths)} | Test: {len(test_paths)}')

X_train, y_train = load_dataset(train_paths, train_labels, 'Train')
X_test,  y_test  = load_dataset(test_paths,  test_labels,  'Test ')
print(f'X_train {X_train.shape} | X_test {X_test.shape}')


## Variasi Hyperparameter (16 Arsitektur)

| Variasi | Nilai |
|---|---|
| Jumlah conv layer | 1, 2 |
| Jumlah filter | 32, 64 |
| Ukuran filter | 3×3, 5×5 |
| Jenis pooling | Max, Average |

In [ ]:
CONFIGS = list(itertools.product(
    [1, 2],          # n_conv_layers
    [32, 64],        # n_filters
    [3, 5],          # kernel_size
    ['max', 'avg'],  # pooling
))

print(f'Total konfigurasi: {len(CONFIGS)}')
for i, (nl, nf, ks, pl) in enumerate(CONFIGS):
    print(f'  [{i+1:2d}] layers={nl}, filters={nf}, kernel={ks}x{ks}, pooling={pl}')


## Training 16 Model

In [ ]:
def build_model(n_layers, n_filters, kernel_size, pooling):
    Pool = MaxPooling2D if pooling == 'max' else AveragePooling2D
    m = Sequential()
    for i in range(n_layers):
        kwargs = dict(filters=n_filters, kernel_size=(kernel_size, kernel_size),
                      activation='relu', padding='same')
        if i == 0:
            kwargs['input_shape'] = (IMG_SIZE[0], IMG_SIZE[1], 3)
        m.add(Conv2D(**kwargs))
        m.add(Pool((2, 2)))
    m.add(Flatten())
    m.add(Dense(128, activation='relu'))
    m.add(Dense(len(CLASSES), activation='softmax'))
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

results   = []
histories = []
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

for idx, (nl, nf, ks, pl) in enumerate(CONFIGS):
    tag       = f'l{nl}_f{nf}_k{ks}_{pl}'
    save_path = f'saved_models/model_{tag}.keras'
    print(f'\n[{idx+1:2d}/16] layers={nl}  filters={nf}  kernel={ks}  pooling={pl}')

    model = build_model(nl, nf, ks, pl)
    hist  = model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE,
                      validation_split=0.1, callbacks=[early_stop], verbose=1)

    model.save(save_path)
    histories.append(hist.history)

    preds = np.argmax(model.predict(X_test, verbose=0), axis=-1)
    f1    = f1_score(y_test, preds, average='macro')

    results.append({'tag': tag, 'n_layers': nl, 'n_filters': nf,
                    'kernel_size': ks, 'pooling': pl,
                    'f1': f1, 'n_params': model.count_params()})
    print(f'    Macro F1: {f1:.4f} | Params: {model.count_params():,}')

with open('saved_models/results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nSemua model tersimpan.')


## Ringkasan Hasil

In [ ]:
results_sorted = sorted(results, key=lambda r: r['f1'], reverse=True)

header = f"{'Rank':<5} {'Tag':<22} {'Layers':>6} {'Filters':>8} {'Kernel':>7} {'Pooling':>8} {'F1':>8} {'Params':>12}"
print(header)
print('-' * len(header))
for rank, r in enumerate(results_sorted, 1):
    print(f"{rank:<5} {r['tag']:<22} {r['n_layers']:>6} {r['n_filters']:>8} "
          f"{r['kernel_size']:>7} {r['pooling']:>8} {r['f1']:>8.4f} {r['n_params']:>12,}")

best = results_sorted[0]
print(f"\nModel terbaik: {best['tag']}  —  Macro F1: {best['f1']:.4f}")


## Grafik Training/Validation Loss per Variasi

In [ ]:
def plot_group(title, groups):
    n = len(groups)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 4))
    if n == 1: axes = [axes]
    fig.suptitle(title, fontsize=13)
    for ax, (label, idxs) in zip(axes, groups):
        for i in idxs:
            nl, nf, ks, pl = CONFIGS[i]
            ax.plot(histories[i]['val_loss'], label=f'l{nl} f{nf} k{ks} {pl}', alpha=0.7)
        ax.set_title(label); ax.set_xlabel('Epoch'); ax.set_ylabel('Val Loss')
        ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()

idx_by = lambda key, val: [i for i, c in enumerate(CONFIGS) if c[['n_layers','n_filters','kernel_size','pooling'].index(key)] == val]

plot_group('Pengaruh Jumlah Layer', [
    ('1 Conv Layer',  [i for i,(nl,*_) in enumerate(CONFIGS) if nl==1]),
    ('2 Conv Layers', [i for i,(nl,*_) in enumerate(CONFIGS) if nl==2]),
])
plot_group('Pengaruh Jumlah Filter', [
    ('32 Filters', [i for i,(_,nf,*_) in enumerate(CONFIGS) if nf==32]),
    ('64 Filters', [i for i,(_,nf,*_) in enumerate(CONFIGS) if nf==64]),
])
plot_group('Pengaruh Ukuran Filter', [
    ('Kernel 3×3', [i for i,(_,_,ks,_) in enumerate(CONFIGS) if ks==3]),
    ('Kernel 5×5', [i for i,(_,_,ks,_) in enumerate(CONFIGS) if ks==5]),
])
plot_group('Pengaruh Jenis Pooling', [
    ('Max Pooling', [i for i,(*_,pl) in enumerate(CONFIGS) if pl=='max']),
    ('Avg Pooling', [i for i,(*_,pl) in enumerate(CONFIGS) if pl=='avg']),
])
